### 3.3.2 Pre-trained Autoencoder (notebook 04)

### 3.3.2.1 Inspiration for our approach
The baseline convolutional autoencoder was limited by the small size of the training set (224 normal images), resulting in poor generalization and weak anomaly detection. To address this, we leveraged transfer learning by incorporating a pretrained ResNet18 encoder, motivated by the strong generalization capabilities of features learned from large-scale datasets like ImageNet. The expectation was that these features would provide richer, more robust representations, potentially enabling the model to better distinguish between normal and anomalous cable regions despite limited task-specific data. This approach is well-established in anomaly detection literature, especially for small or imbalanced datasets.

<div style="text-align: center;"> <img src="../figures/Fig20_ResNet_18.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 20. ResNet18 architecture used as the encoder for the pre-trained autoencoder.</i> <br> </div>

### 3.3.2.2 Pretrained autoencoder
The improved model architecture replaces the custom encoder with the first layers of a ResNet18 network pretrained on ImageNet. The encoder outputs feature maps, which are processed by a bottleneck convolutional block. The decoder consists of several transposed convolutional stages, progressively upsampling the features back to the original image size (256×256). The final output layer uses a sigmoid activation to produce a reconstructed image.

We explored two training strategies:

Frozen encoder: The ResNet18 encoder weights were kept fixed, training only the bottleneck and decoder. This configuration acted as a strong regularizer, leveraging the pretrained features.
Fine-tuned encoder: The top layers of the encoder were unfrozen and fine-tuned on the cable dataset. However, this approach did not yield meaningful improvements.

**Training and Loss Curves:**
Figure 21 shows the training and validation loss curves for the pre-trained autoencoder with a frozen encoder, while Figure 26 presents the loss curves for the fine-tuned variant.

<div style="text-align: center;"> <img src="../figures/Fig21_Pretrained_Loss.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 21. Training and validation loss curves for the pre-trained autoencoder (frozen encoder).</i> <br> </div> 

<div style="text-align: center;"> <img src="../figures/Fig26_Pretrained_Finetuned_Loss.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 26. Training and validation loss curves for the fine-tuned encoder variant.</i> <br> </div>

**Qualitative Results:**
Figures 22 and 23 provide visual evidence of the model’s reconstruction ability. For normal images (Figure 22), the pre-trained autoencoder generally preserves the overall cable structure, but some fine details may be lost. For defect images (Figure 23), reconstructions often fail to accurately restore the anomalous regions, indicating the model’s difficulty in handling complex defects. Figure 25 shows anomaly maps, where the model highlights anomalous regions at the pixel level.

<div style="text-align: center;"> <img src="../figures/Fig22_Pretrained_Normal_Reconstruction.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 22. Reconstruction results for normal images (pre-trained encoder).</i> <br> </div> 

<div style="text-align: center;"> <img src="../figures/Fig23_Pretrained_Defect_Reconstruction.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 23. Reconstruction results for defect images (pre-trained encoder).</i> <br> </div> 

<div style="text-align: center;"> <img src="../figures/Fig25_Pretrained_Anomaly_Maps.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 25. Anomaly map visualizations for the pre-trained autoencoder.</i> <br> </div>

**Fine-tuning Results:**
Figure 27 shows that fine-tuning the encoder does not noticeably enhance reconstruction quality for normal images, with results similar to the frozen encoder. Figures 28 and 29 illustrate that varying the learning rate and the number of unfrozen layers has minimal positive impact, and in some cases, reconstructions may even degrade, reinforcing that fine-tuning is not beneficial in this context.

<div style="text-align: center;"> <img src="../figures/Fig27_Finetuned_Normal_Reconstruction.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 27. Normal image reconstructions after fine-tuning.</i> <br> </div> 

<div style="text-align: center;"> <img src="../figures/Fig28_LR=1e-4_Reconstructions.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 28. Reconstructions with learning rate 1e-4.</i> <br> </div> 

<div style="text-align: center;"> <img src="../figures/Fig29_Layers_3_&_4_Reconstructions.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 29. Reconstructions with different numbers of unfrozen layers.</i> <br> </div>

**Quantitative Results:**
The ROC curves in Figure 24 confirm that the pre-trained encoder substantially improves pixel-level anomaly localization (Pixel AUROC and AUPRO) compared to the baseline. However, image-level anomaly detection (Image AUROC) does not improve, indicating that while local anomalies are better detected, overall image classification remains challenging. This is consistent with the observed reconstruction limitations.

<div style="text-align: center;"> <img src="../figures/Fig24_Pretrained_ROC_Curves.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 24. ROC curves for pixel- and image-level anomaly detection (pre-trained encoder).</i> <br> </div>

**High-Quality Reconstructions:**
Figure 30 highlights examples with high SSIM, demonstrating cases where the model produced good reconstructions.

<div style="text-align: center;"> <img src="../figures/Fig30_High_SSIM_Reconstructions.png" style="display: block; margin: 0 auto; max-width: 60%;"> <br> <i>Figure 30. Examples of high-quality reconstructions (high SSIM).</i> <br> </div>

The table below summarizes the key metrics (from notebook section 8.1):

| Metric        | Baseline 3-Block | Pre-Trained (Frozen) | Fine-Tuned (L4) |
|-------------- |-----------------:|---------------------:|----------------:|
| Image AUROC   | 0.5547           | 0.4670               | 0.4299          |
| Pixel AUROC   | 0.6937           | 0.8114               | 0.8112          |
| AUPRO         | 0.3625           | 0.4313               | 0.3877          |

**Results Summary**

The quantitative results in the table above demonstrate that incorporating a pre-trained ResNet18 encoder led to a substantial improvement in pixel-level anomaly localization, as shown by higher Pixel AUROC (0.8114 vs. 0.6937) and AUPRO (0.4313 vs. 0.3625) compared to the baseline 3-block autoencoder. However, image-level anomaly detection performance (Image AUROC) decreased with the pre-trained encoder (0.4670 vs. 0.5547). Fine-tuning the top layers of the encoder (L4) did not yield further improvements and, in fact, resulted in a slight decrease in all metrics compared to the frozen variant.

The pre-trained encoder improved pixel-level metrics but not image-level anomaly detection. Fine-tuning did not help and sometimes worsened results. The architecture struggled to reconstruct the internal cable structure, limiting overall anomaly detection performance. Further architectural changes are needed for substantial improvement.